# Preprocess the prostate MRI dataset

Run this notebook in Google Colab with a GPU runtime if desired. It mounts Google Drive, clones the project into Colab's local storage, extracts the dataset locally, and stores processed outputs on Google Drive.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

project_dir = Path('/content/Masters_quad_joined_prostate_detection')
project_parent = project_dir.parent
repository_url = 'https://github.com/Peter-worm/Masters_quad_joined_prostate_detection.git'

if not (project_dir / '.git').is_dir():
    if project_dir.exists():
        raise FileNotFoundError(
            f'{project_dir} exists but is not a Git checkout. '
            'Move it or update project_dir before running this notebook.'
        )
    project_parent.mkdir(parents=True, exist_ok=True)
    clone_result = subprocess.run(
        ['git', 'clone', '--depth', '1', repository_url, str(project_dir)],
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            f'Unable to clone {repository_url}:\n{clone_result.stderr.strip()}'
        )

os.chdir(project_dir)
print(f'Project directory: {project_dir}')


## Optional: update the local checkout

Run the next cell after pushing code changes to GitHub and before running imports. Set `UPDATE_REPOSITORY` to `False` when you want to keep the current checkout.

In [ ]:
UPDATE_REPOSITORY = True

if UPDATE_REPOSITORY and (project_dir / '.git').is_dir():
    update_result = subprocess.run(
        ['git', '-C', str(project_dir), 'pull', '--ff-only'],
        capture_output=True,
        text=True,
    )
    if update_result.returncode != 0:
        raise RuntimeError(
            'Unable to update the local checkout. Commit or discard local '
            f'changes, then retry:\n{update_result.stderr.strip()}'
        )
    print(update_result.stdout.strip() or 'Repository is already up to date.')


In [ ]:
%pip install -r requirements.txt


In [ ]:
preprocessing_path = project_dir / 'preprocessing'
if str(preprocessing_path) not in sys.path:
    sys.path.insert(0, str(preprocessing_path))


In [ ]:
from preprocessing.settings.main_settings import test_settings

settings = test_settings().get_setting_dictionary()
dataset_dir = Path(settings['original_data_folder'])
dataset_archive = Path(settings['raw_data_archive'])

if not dataset_dir.is_dir():
    if not dataset_archive.is_file():
        raise FileNotFoundError(f'Dataset archive not found: {dataset_archive}')
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['unzip', '-q', str(dataset_archive), '-d', str(dataset_dir.parent)],
        check=True,
    )
    print(f'Dataset extracted to {dataset_dir}')
else:
    print(f'Dataset already exists at {dataset_dir}; skipping extraction.')


## Run preprocessing


In [ ]:
from preprocessing.core.file_manager import preprocess_file_manager
from preprocessing.core.preprocessing_pipeline import run_preprocessing_pipeline

file_manager = preprocess_file_manager(
    settings['preprocessed_data_path'],
    settings['preprocessed_steps'],
    settings['channels'],
)

run_preprocessing_pipeline(
    settings=settings,
    file_manager=file_manager,
    patient_filter=None,
)


## Archive processed splits

The training notebook extracts these archives into Colab's local storage before training.

In [ ]:
import tarfile

processed_dir = Path(settings['preprocessed_data_path'])
archive_dir = Path(settings['preprocessed_archive_path'])
archive_dir.mkdir(parents=True, exist_ok=True)

for split in ('train', 'test'):
    split_dir = processed_dir / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f'Processed split not found: {split_dir}')
    
    pickle_files = sorted(split_dir.glob('*.pkl'))
    if not pickle_files:
        raise FileNotFoundError(f'No pickle files found in {split_dir}')
    
    archive_path = archive_dir / f'processed_{split}.tar.gz'
    with tarfile.open(archive_path, 'w:gz') as archive:
        for pickle_file in pickle_files:
            archive.add(pickle_file, arcname=pickle_file.name)

    print(f'Created {archive_path} ({len(pickle_files)} files)')
